# Gold: Weather Monthly Summary (v1)

Reads the `noaa_weather.daily_observations` bronze table (see
`data-generators/bronze_ingest_noaa_weather.ipynb`), replaces NOAA's
missing-value sentinels (`9999.9` / `999.9` / `99.99`) with real nulls, and
aggregates by month: avg/max/min temperature, total precipitation, and
average wind speed. Persists the result as a managed Spark table for demo
dashboards/queries.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "noaa_weather"
SOURCE_TABLE = "daily_observations"
TARGET_TABLE = "gold_weather_monthly_summary_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Clean NOAA missing-value sentinels

GSOD uses fixed sentinel values instead of nulls for missing readings:
`9999.9` for temperature fields, `999.9` for wind speed, `99.99` for
precipitation. Replace them with real nulls before aggregating so they
don't skew averages.

In [ ]:
clean_df = (
    df
    .withColumn("TEMP", F.when(F.col("TEMP") == 9999.9, None).otherwise(F.col("TEMP")))
    .withColumn("MAX", F.when(F.col("MAX") == 9999.9, None).otherwise(F.col("MAX")))
    .withColumn("MIN", F.when(F.col("MIN") == 9999.9, None).otherwise(F.col("MIN")))
    .withColumn("WDSP", F.when(F.col("WDSP") == 999.9, None).otherwise(F.col("WDSP")))
    .withColumn("PRCP", F.when(F.col("PRCP") == 99.99, None).otherwise(F.col("PRCP")))
    .withColumn("month", F.date_format(F.to_date("DATE"), "yyyy-MM"))
)

## Transform: monthly weather summary

In [ ]:
summary_df = (
    clean_df
    .groupBy("month")
    .agg(
        F.round(F.avg("TEMP"), 1).alias("avg_temp_f"),
        F.round(F.max("MAX"), 1).alias("max_temp_f"),
        F.round(F.min("MIN"), 1).alias("min_temp_f"),
        F.round(F.sum("PRCP"), 2).alias("total_precip_in"),
        F.round(F.avg("WDSP"), 1).alias("avg_wind_speed_kt"),
    )
    .orderBy("month")
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_months FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY month LIMIT 12").show()